# Multilingual Transformer Classifier

In this notebook an XLM-RoBERTa large (number or parameters: 550M) model is fine-tuned for classification. This model is trained on 100+ languages and can be used for text classification in any of these languages.

It has been chosen because it was trained in a large amount of data from general domain such as what is expected in this dataset. Also it has been trained on multiple languages which is a good fit for this dataset.

In [1]:
import torch
import numpy as np
from transformers import XLMRobertaForSequenceClassification, XLMRobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from transformers import EarlyStoppingCallback
import transformers
import pandas as pd
from datasets import Dataset
import json

# transformers.logging.set_verbosity_error()



MODEL_NAME = 'xlm-roberta-large'

In [2]:
import torch
from torch.optim.lr_scheduler import LambdaLR
import os

def train_xlm_roberta_on_fold(model_id, train_ds, val_ds):
    os.environ["CUDA_VISIBLE_DEVICES"]="0"
    model = XLMRobertaForSequenceClassification.from_pretrained(model_id, num_labels=2)
    tokenizer = XLMRobertaTokenizer.from_pretrained(model_id)
    
    def tokenize_function(example):
        return tokenizer(example["text_pair"][0], example["text_pair"][1], truncation=True)
    
    train_ds = train_ds.map(tokenize_function)
    val_ds = val_ds.map(tokenize_function)
    
    args = TrainingArguments(
        output_dir="nbs/other/model",
        eval_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=15,
        logging_dir="nbs/other/logs",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {
            "f1": f1_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds),
            "roc_auc": roc_auc_score(labels, logits[:, 1]),
            # "learning_rate": optimizer.param_groups[0]["lr"],
        }

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
    
    # Calculate the total number of training steps
    num_training_steps = (len(train_ds) // args.per_device_train_batch_size) * args.num_train_epochs

    # Define a lambda function for the linear learning rate decay
    lr_lambda = lambda step: 1 - (step / num_training_steps)
    
    # Create the scheduler using LambdaLR
    lr_scheduler = LambdaLR(optimizer, lr_lambda)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        optimizers=[optimizer, lr_scheduler],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )

    trainer.train()
    
    return trainer


In [3]:
df = pd.read_csv("nbs/other/claim_matching_dataset.csv")
df = df.fillna("")

unverified_claims = df["unverified claim"].tolist()
reviewed_claims = df["reviewed claim"].tolist()
labels = df["similarity"].tolist()

data = pd.DataFrame({
    "text_pair": list(zip(unverified_claims, reviewed_claims)),
    "label": labels
})

d_splits = json.load(open("nbs/other/splits.json"))

ds_train = Dataset.from_pandas(data.iloc[d_splits["train"]])
ds_val = Dataset.from_pandas(data.iloc[d_splits["val"]])
ds_test = Dataset.from_pandas(data.iloc[d_splits["test"]])


In [4]:
trainer = train_xlm_roberta_on_fold("xlm-roberta-large", ds_train, ds_val)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/3119 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Map:   0%|          | 0/356 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Roc Auc
1,0.534500,0.533547,0.000000,0.000000,0.000000,0.869127
2,0.463700,0.400639,0.000000,0.000000,0.000000,0.902592
3,0.299200,0.348641,0.763636,0.851351,0.692308,0.921626
4,0.203100,0.301092,0.771739,0.763441,0.780220,0.927348
5,0.167900,0.276154,0.860335,0.875000,0.846154,0.945843
6,0.136200,0.305023,0.819277,0.906667,0.747253,0.957122
7,0.127300,0.472139,0.800000,0.891892,0.725275,0.931785
8,0.110900,0.340090,0.847059,0.911392,0.791209,0.948580
9,0.112300,0.435035,0.821429,0.896104,0.758242,0.935061
10,0.076700,0.522448,0.807453,0.928571,0.714286,0.940079


In [5]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")

def tokenize_function(example):
    return tokenizer(example["text_pair"][0], example["text_pair"][1], truncation=True)

ds_train = ds_train.map(tokenize_function)
ds_val = ds_val.map(tokenize_function)
ds_test = ds_test.map(tokenize_function)

Map:   0%|          | 0/3119 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Map:   0%|          | 0/356 [00:00<?, ? examples/s]

Map:   0%|          | 0/390 [00:00<?, ? examples/s]

In [6]:
y_pred_val_prob = torch.tensor(trainer.predict(ds_val).predictions).softmax(dim=1)
y_pred_val_prob_1 = y_pred_val_prob[:, 1].numpy()
y_pred_val = np.argmax(y_pred_val_prob, axis=1)

from sklearn.metrics import classification_report

print(classification_report(ds_val["label"], y_pred_val))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       265
           1       0.88      0.85      0.86        91

    accuracy                           0.93       356
   macro avg       0.91      0.90      0.91       356
weighted avg       0.93      0.93      0.93       356



In [7]:
y_pred_test_prob = torch.tensor(trainer.predict(ds_test).predictions).softmax(dim=1)
y_pred_test_prob_1 = y_pred_test_prob[:, 1].numpy()
y_pred_test = np.argmax(y_pred_test_prob, axis=1)

from sklearn.metrics import classification_report

print(classification_report(ds_test["label"], y_pred_test))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94       284
           1       0.80      0.88      0.84       106

    accuracy                           0.91       390
   macro avg       0.88      0.90      0.89       390
weighted avg       0.91      0.91      0.91       390



In [8]:
# Save the model (including weights, tokenizer, and config)
trainer.save_model("nbs/other/trf_classifier")

# If tokenizer is not saved automatically, explicitly save it
tokenizer.save_pretrained("nbs/other/trf_classifier")

('nbs/other/trf_classifier/tokenizer_config.json',
 'nbs/other/trf_classifier/special_tokens_map.json',
 'nbs/other/trf_classifier/sentencepiece.bpe.model',
 'nbs/other/trf_classifier/added_tokens.json')

In [16]:
y_pred_train_prob = torch.tensor(trainer.predict(ds_train).predictions).softmax(dim=1)
y_pred_train_prob_1 = y_pred_train_prob[:, 1].numpy()
y_pred_train_prob_1

/gpfs/projects/bsc14/scratch/.conda/factcheck/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


array([0.01014117, 0.98899186, 0.98662347, ..., 0.9554574 , 0.00251236,
       0.00465853], dtype=float32)

In [17]:
df_preds = df[['url', 'unverified claim', 'reviewed claim', 'similarity']].copy()
df_preds.loc[d_splits["train"], "trf_preds"]= y_pred_train_prob_1
df_preds.loc[d_splits["val"], "trf_preds"]= y_pred_val_prob_1
df_preds.loc[d_splits["test"], "trf_preds"]= y_pred_test_prob_1
df_preds.to_csv("nbs/other/trf_classifier/trf_predictions.csv", index=False)